# FinShield Fraud Detection - Exploratory Data Analysis

**Author:** Aman  
**Date:** November 2026  
**Dataset:** IEEE-CIS Fraud Detection

This notebook explores the fraud detection dataset to understand patterns, distributions, and key features.

In [ ]:
# Import libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully!')

## 1. Load Data

In [ ]:
# Load dataset
from finshield.data import load_transaction_data

df = load_transaction_data()
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## 2. Dataset Overview

In [ ]:
# Basic info
print(f"Total Transactions: {len(df):,}")
print(f"Total Features: {df.shape[1]}")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nData Types:")
print(df.dtypes.value_counts())

## 3. Target Variable Analysis

In [ ]:
# Fraud distribution
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100

print("Fraud Distribution:")
print(f"  Non-Fraud: {fraud_counts[0]:,} ({fraud_pct[0]:.2f}%)")
print(f"  Fraud:     {fraud_counts[1]:,} ({fraud_pct[1]:.2f}%)")
print(f"\nClass Imbalance Ratio: {fraud_counts[0] / fraud_counts[1]:.2f}:1")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
axes[0].bar(['Non-Fraud', 'Fraud'], fraud_counts.values, color=['green', 'red'], alpha=0.7)
axes[0].set_ylabel('Count')
axes[0].set_title('Fraud vs Non-Fraud Transactions')
axes[0].grid(alpha=0.3)

# Percentage plot
axes[1].pie(fraud_counts.values, labels=['Non-Fraud', 'Fraud'], autopct='%1.2f%%', 
            colors=['green', 'red'], startangle=90)
axes[1].set_title('Fraud Distribution (%)')

plt.tight_layout()
plt.show()

## 4. Missing Values Analysis

In [ ]:
# Calculate missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_pct
}).sort_values('Missing_Percentage', ascending=False)

# Filter columns with >50% missing
high_missing = missing_df[missing_df['Missing_Percentage'] > 50]

print(f"Columns with >50% missing values: {len(high_missing)}")
print(f"\nTop 10 columns with most missing data:")
print(high_missing.head(10))

# Visualize
plt.figure(figsize=(12, 6))
top_missing = missing_df.head(20)
plt.barh(range(len(top_missing)), top_missing['Missing_Percentage'])
plt.yticks(range(len(top_missing)), top_missing.index)
plt.xlabel('Missing Percentage (%)')
plt.title('Top 20 Features with Most Missing Values')
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Transaction Amount Analysis

In [ ]:
# Transaction amount statistics
print("Transaction Amount Statistics:")
print(df.groupby('isFraud')['TransactionAmt'].describe())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot
df.boxplot(column='TransactionAmt', by='isFraud', ax=axes[0])
axes[0].set_ylabel('Transaction Amount ($)')
axes[0].set_xlabel('Is Fraud')
axes[0].set_title('Transaction Amount by Fraud Status')
axes[0].set_yscale('log')

# Distribution plot
for fraud_val in [0, 1]:
    data = df[df['isFraud'] == fraud_val]['TransactionAmt']
    axes[1].hist(data, bins=50, alpha=0.5, label=f'Fraud={fraud_val}')
axes[1].set_xlabel('Transaction Amount ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Transaction Amount Distribution')
axes[1].set_xscale('log')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Time-based Analysis

In [ ]:
# Transaction time analysis
print("Transaction Time Range:")
print(f"  Min: {df['TransactionDT'].min()}")
print(f"  Max: {df['TransactionDT'].max()}")
print(f"  Range: {df['TransactionDT'].max() - df['TransactionDT'].min()} seconds")
print(f"  Range: {(df['TransactionDT'].max() - df['TransactionDT'].min()) / (24*3600):.0f} days")

# Fraud rate over time
df_sorted = df.sort_values('TransactionDT')
window_size = 10000
fraud_rate_rolling = df_sorted['isFraud'].rolling(window=window_size).mean() * 100

plt.figure(figsize=(14, 5))
plt.plot(range(len(fraud_rate_rolling)), fraud_rate_rolling, alpha=0.7)
plt.xlabel('Transaction Index (Time-ordered)')
plt.ylabel('Fraud Rate (%) - Rolling Average')
plt.title(f'Fraud Rate Over Time (Window={window_size:,})')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Product Code Analysis

In [ ]:
# Product code distribution
product_fraud = df.groupby('ProductCD')['isFraud'].agg(['count', 'sum', 'mean']).sort_values('mean', ascending=False)
product_fraud.columns = ['Total', 'Fraud_Count', 'Fraud_Rate']
product_fraud['Fraud_Rate'] = product_fraud['Fraud_Rate'] * 100

print("Fraud Rate by Product Code:")
print(product_fraud)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Transaction count
product_fraud['Total'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_ylabel('Transaction Count')
axes[0].set_title('Transactions by Product Code')
axes[0].grid(alpha=0.3)

# Fraud rate
product_fraud['Fraud_Rate'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Product Code')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Card Features Analysis

In [ ]:
# Card features correlation with fraud
card_features = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6']

print("Card Features Summary:")
for feat in card_features:
    if feat in df.columns:
        unique_count = df[feat].nunique()
        missing_pct = (df[feat].isnull().sum() / len(df)) * 100
        print(f"  {feat}: {unique_count:,} unique values, {missing_pct:.1f}% missing")

## 9. Feature Correlation Analysis

In [ ]:
# Select numeric features for correlation
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove TransactionID and target
numeric_features = [f for f in numeric_features if f not in ['TransactionID', 'isFraud']]

# Sample for faster computation
sample_df = df[numeric_features + ['isFraud']].sample(n=min(10000, len(df)), random_state=42)

# Correlation with target
correlations = sample_df.corr()['isFraud'].drop('isFraud').sort_values(ascending=False)

print("Top 20 features most correlated with fraud:")
print(correlations.head(20))

# Visualize
plt.figure(figsize=(10, 8))
top_corr = correlations.head(20)
plt.barh(range(len(top_corr)), top_corr.values)
plt.yticks(range(len(top_corr)), top_corr.index)
plt.xlabel('Correlation with Fraud')
plt.title('Top 20 Features Correlated with Fraud')
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Key Insights & Summary

### Key Findings:

1. **Severe Class Imbalance**: Only 3.5% of transactions are fraudulent (27:1 ratio)
2. **High Missing Data**: 174 columns have >50% missing values
3. **Transaction Amount**: Fraud transactions show different amount distributions
4. **Temporal Patterns**: Fraud rate varies over time
5. **Product Variation**: Different products have different fraud rates

### Recommendations for Modeling:

1. **Handle Imbalance**: Use scale_pos_weight or SMOTE
2. **Missing Data**: Median imputation for numeric, 'missing' category for categorical
3. **Time-aware Split**: Use TransactionDT for train/test split
4. **Feature Engineering**: Create aggregations and interactions
5. **Evaluation Metrics**: Focus on ROC-AUC, Recall, and Precision (not accuracy)

## 11. Export Summary Statistics

In [ ]:
# Create summary report
summary = {
    'Total_Transactions': len(df),
    'Total_Features': df.shape[1],
    'Fraud_Count': df['isFraud'].sum(),
    'Fraud_Rate': f"{df['isFraud'].mean()*100:.2f}%",
    'Missing_Values': df.isnull().sum().sum(),
    'Missing_Percentage': f"{(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]))*100:.2f}%",
    'High_Missing_Columns': len(missing_df[missing_df['Missing_Percentage'] > 50])
}

print("\n" + "="*60)
print("EDA SUMMARY")
print("="*60)
for key, value in summary.items():
    print(f"{key.replace('_', ' ')}: {value}")
print("="*60)

print("\n✅ EDA Complete!")